In [2]:
import pandas as pd

data = pd.read_csv("data_clean.csv")  # твой CSV файл
print(data.head())



   from_lat   from_lon     to_lat     to_lon  predicted_time_min
0  47.21805  39.697207  47.217621  39.707611            5.684958
1  47.21805  39.697207  47.221096  39.690970            3.639299
2  47.21805  39.697207  47.225482  39.717826            8.598251
3  47.21805  39.697207  47.255549  39.644401           22.955914
4  47.21805  39.697207  47.224904  39.711349            5.890577


In [3]:
import torch
from torch_geometric.data import Data

# Преобразуем координаты в узлы
nodes = {}
node_id = 0
edges = []
edge_weights = []

for _, row in data.iterrows():
    f_node = (row['from_lat'], row['from_lon'])
    t_node = (row['to_lat'], row['to_lon'])
    
    if f_node not in nodes:
        nodes[f_node] = node_id
        node_id += 1
    if t_node not in nodes:
        nodes[t_node] = node_id
        node_id += 1
    
    edges.append([nodes[f_node], nodes[t_node]])
    edge_weights.append(row['predicted_time_min'])

# Преобразуем в тензоры
edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
edge_attr = torch.tensor(edge_weights, dtype=torch.float).view(-1, 1)
x = torch.tensor(list(nodes.keys()), dtype=torch.float)  # координаты узлов

data_graph = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)


c:\Users\vladulik\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GraphConv

class TravelTimeGNN(torch.nn.Module):
    def __init__(self, input_dim=2, hidden_dim=32):
        super().__init__()
        self.conv1 = GraphConv(input_dim, hidden_dim)
        self.conv2 = GraphConv(hidden_dim, hidden_dim)
        self.fc = torch.nn.Linear(hidden_dim, 1)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        out = self.fc(x)
        return out


In [11]:
# ===============================
# 1. Импорты
# ===============================
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GraphConv
import pandas as pd
import networkx as nx
import folium

# ===============================
# 2. Загрузка данных
# ===============================
# Убедись, что рядом есть файл routes.csv
data = pd.read_csv("data_clean.csv")  # колонки: from_lat,from_lon,to_lat,to_lon,predicted_time_min

# ===============================
# 3. Построение графа
# ===============================
nodes = {}
node_id = 0
edges = []
edge_weights = []

for _, row in data.iterrows():
    f_node = (row['from_lat'], row['from_lon'])
    t_node = (row['to_lat'], row['to_lon'])
    
    if f_node not in nodes:
        nodes[f_node] = node_id
        node_id += 1
    if t_node not in nodes:
        nodes[t_node] = node_id
        node_id += 1
    
    edges.append([nodes[f_node], nodes[t_node]])
    edge_weights.append(row['predicted_time_min'])

edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
edge_attr = torch.tensor(edge_weights, dtype=torch.float).view(-1, 1)
x = torch.tensor(list(nodes.keys()), dtype=torch.float)  # координаты узлов

data_graph = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

# ===============================
# 4. Модель GNN
# ===============================
class TravelTimeGNN(torch.nn.Module):
    def __init__(self, input_dim=2, hidden_dim=32):
        super().__init__()
        self.conv1 = GraphConv(input_dim, hidden_dim)
        self.conv2 = GraphConv(hidden_dim, hidden_dim)
        self.fc = torch.nn.Linear(hidden_dim, 1)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        out = self.fc(x)
        return F.relu(out)  # гарантируем, что время >= 0

# ===============================
# 5. Обучение модели
# ===============================
model = TravelTimeGNN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = torch.nn.MSELoss()

print("Начинаем обучение...")
for epoch in range(300):
    model.train()
    optimizer.zero_grad()
    out = model(data_graph)
    pred_time = (out[data_graph.edge_index[0]] + out[data_graph.edge_index[1]]) / 2
    loss = loss_fn(pred_time, data_graph.edge_attr)
    loss.backward()
    optimizer.step()
    if epoch % 50 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

print("✅ Обучение завершено.")

# ===============================
# 6. Сохранение модели
# ===============================
torch.save(model.state_dict(), "gnn_route_model.pt")
print("💾 Модель сохранена как 'gnn_route_model.pt'")

# ===============================
# 7. Загрузка модели
# ===============================
loaded_model = TravelTimeGNN()
loaded_model.load_state_dict(torch.load("gnn_route_model.pt"))
loaded_model.eval()
print("📂 Модель загружена для тестирования")

# ===============================
# 8. Предсказание и построение графа
# ===============================
with torch.no_grad():
    out = loaded_model(data_graph)
    pred_time = ((out[data_graph.edge_index[0]] + out[data_graph.edge_index[1]]) / 2).squeeze().numpy()

# защита от отрицательных весов
pred_time = abs(pred_time)

# NetworkX граф
G = nx.DiGraph()
for i, (u, v) in enumerate(data_graph.edge_index.t().tolist()):
    G.add_edge(u, v, weight=float(pred_time[i]))

# ===============================
# 9. Поиск кратчайшего пути
# ===============================
start_coords = (47.21805, 39.697207)
end_coords = (47.226024, 39.729858)

start_node = nodes[start_coords]
end_node = nodes[end_coords]

path = nx.shortest_path(G, source=start_node, target=end_node, weight='weight')
total_time = nx.shortest_path_length(G, source=start_node, target=end_node, weight='weight')

inv_nodes = {v: k for k, v in nodes.items()}
coords_path = [inv_nodes[p] for p in path]

print("\n🧭 Кратчайший путь по узлам:", path)
print("⏱ Ожидаемое время (мин):", round(total_time, 2))
print("\n📍 Маршрут (координаты):")
for lat, lon in coords_path:
    print(f"({lat:.6f}, {lon:.6f})")

# ===============================
# 10. Визуализация маршрута на карте
# ===============================
mid_lat = sum(lat for lat, _ in coords_path) / len(coords_path)
mid_lon = sum(lon for _, lon in coords_path) / len(coords_path)
m = folium.Map(location=[mid_lat, mid_lon], zoom_start=13)

# Добавим линии маршрута
folium.PolyLine(coords_path, color="blue", weight=5, opacity=0.7).add_to(m)

# Маркеры начала и конца
folium.Marker(coords_path[0], popup="Start", icon=folium.Icon(color="green")).add_to(m)
folium.Marker(coords_path[-1], popup="End", icon=folium.Icon(color="red")).add_to(m)

m.save("route_map.html")
print("\n🗺 Карта сохранена в файл route_map.html — открой его в браузере 🚀")


Начинаем обучение...
Epoch 0 | Loss: 240.7701
Epoch 50 | Loss: 240.7701
Epoch 100 | Loss: 240.7701
Epoch 150 | Loss: 240.7701
Epoch 200 | Loss: 240.7701
Epoch 250 | Loss: 240.7701
✅ Обучение завершено.
💾 Модель сохранена как 'gnn_route_model.pt'
📂 Модель загружена для тестирования

🧭 Кратчайший путь по узлам: [0, 13]
⏱ Ожидаемое время (мин): 0.0

📍 Маршрут (координаты):
(47.218050, 39.697207)
(47.226024, 39.729858)

🗺 Карта сохранена в файл route_map.html — открой его в браузере 🚀


In [6]:
import networkx as nx

model.eval()
with torch.no_grad():
    out = model(data_graph)
    pred_time = ((out[data_graph.edge_index[0]] + out[data_graph.edge_index[1]]) / 2).squeeze().numpy()

# Построим граф в NetworkX
G = nx.DiGraph()
for i, (u, v) in enumerate(data_graph.edge_index.t().tolist()):
    G.add_edge(u, v, weight=pred_time[i])

# Находим кратчайший путь между узлами
source = 0  # id стартовой точки
target = 5  # id конечной точки

path = nx.shortest_path(G, source=source, target=target, weight='weight')
time = nx.shortest_path_length(G, source=source, target=target, weight='weight')

print("Кратчайший путь:", path)
print("Ожидаемое время:", time)


Кратчайший путь: [0, 5]
Ожидаемое время: 11.570284
